# Assessment 1: Analysing historical data with system performance - Phase 2

## Stefan Garevski (33759839)

# Part A 

## 1. Analytical Query Design and Implementation

**Business Query:
"Which DTP regions have the highest rates of serious or fatal road crashes across different times of day, and how do these rates vary across different road types?"**

This analysis will look at the severity of road crashes across Victoria by serious or fatal crashes across region, time of day, and road type. Specifically, the analysis will quanitfy this by looking at the proportion of crashes that are serious or fatal. This provides a more meaningful measure of crash severity because regions or road types with a larger number of crashes are not automatically the locations with the highest severity rate.

The analysis uses the Victorian Road Crash Data dataset which contains approximately 200,000 crash records so this makes the dataset suitable for demonstrating distributed data processing using Spark. The dataset contains the required attributes for this analysis, including ACCIDENT_NO, ACCIDENT_TIME, DTP_REGION, ROAD_TYPE, SERIOUSINJURY, and FATALITY.


The query implements a multi-stage analytical pipeline consisting of several operations.

Time-based analysis: The ACCIDENT_TIME field is used to extract the hour of each crash, and then transformed into four time periods: overnight, morning, afternoon, and evening. This allows the time of day to be analysed rather than treating time as a filter.

Derived severity measure: A SERIOUS_FATAL_CRASH indicator is created. It is assigned a value of 1 when a crash has one or more serious injuries or fatalities, and 0 otherwise. This derived measure is used to calculate the serious/fatal crash rate.

Multi-level aggregation: Records are grouped simultaneously by DTP_REGION, TIME_PERIOD, and ROAD_TYPE. The aggregation calculates total crashes, serious/fatal crashes, serious injuries, and fatalities. This allows comparison between geographic region, time period, and road type.

Post-aggregation filter: Groups with fewer than 100 crashes are removed post-aggregation. This is so that low volume combinations do not produce unstable or misleading severity rates.

Window function: ROW_NUMBER() window function partitions the aggregated results by DTP_REGION and ranks combinations using SERIOUS_FATAL_RATE in descending order. The top three combinations for each region are then retained, which allows comparison of the highest-rate time and road-type combinations within each region.

Just using a simple WHERE filter would only identify individual records satisfying a condition and could not calculate regional severity rates. Similarly, one GROUP BY would produce aggregate statistics but would not identify and rank the highest rate combinations within each region. Therefore, all these operations are necessary in answering the business question.

The query is also appropriate for distributed processing because the dataset contains approximately 200,000 records and the analysis requires aggregation across more than one variable. Spark can distribute the records across partitions, perform aggregations in parallel, shuffle records according to the grouping variables, and perform the final aggregation with window-based ranking. 

## 2. Dataframe Implementation

In [1]:
#initial imports
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col,
    when,
    hour,
    to_timestamp,
    countDistinct,
    sum,
    round,
    substring,
    row_number,
    spark_partition_id,
    min,
    max,
    avg,
    count
)
from pyspark.sql.window import Window

#initiate spark session
spark = SparkSession.builder \
    .appName("Victorian Road Crash Analysis") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/14 20:06:04 WARN Utils: Your hostname, Stefans-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.115 instead (on interface en0)
26/09/14 20:06:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/anaconda3/envs/ITO5202/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/14 20:06:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
#define schema of variables in dataset 
schema = StructType([
    StructField("ACCIDENT_NO", StringType(), True),
    StructField("ACCIDENT_DATE", StringType(), True),
    StructField("ACCIDENT_TIME", StringType(), True),
    StructField("ACCIDENT_TYPE", StringType(), True),
    StructField("DAY_OF_WEEK", StringType(), True),
    StructField("DCA_CODE", StringType(), True),
    StructField("DCA_CODE_DESCRIPTION", StringType(), True),
    StructField("LIGHT_CONDITION", StringType(), True),
    StructField("POLICE_ATTEND", StringType(), True),
    StructField("ROAD_GEOMETRY", StringType(), True),
    StructField("SEVERITY", StringType(), True),
    StructField("SPEED_ZONE", StringType(), True),
    StructField("RUN_OFFROAD", StringType(), True),
    StructField("ROAD_NAME", StringType(), True),
    StructField("ROAD_TYPE", StringType(), True),
    StructField("ROAD_ROUTE_1", StringType(), True),
    StructField("LGA_NAME", StringType(), True),
    StructField("DTP_REGION", StringType(), True),
    StructField("LATITUDE", DoubleType(), True),
    StructField("LONGITUDE", DoubleType(), True),
    StructField("VICGRID_X", DoubleType(), True),
    StructField("VICGRID_Y", DoubleType(), True),
    StructField("TOTAL_PERSONS", IntegerType(), True),
    StructField("INJ_OR_FATAL", IntegerType(), True),
    StructField("FATALITY", IntegerType(), True),
    StructField("SERIOUSINJURY", IntegerType(), True),
    StructField("OTHERINJURY", IntegerType(), True),
    StructField("NONINJURED", IntegerType(), True),
    StructField("MALES", IntegerType(), True),
    StructField("FEMALES", IntegerType(), True),
    StructField("BICYCLIST", IntegerType(), True),
    StructField("PASSENGER", IntegerType(), True),
    StructField("DRIVER", IntegerType(), True),
    StructField("PEDESTRIAN", IntegerType(), True),
    StructField("PILLION", IntegerType(), True),
    StructField("MOTORCYCLIST", IntegerType(), True),
    StructField("UNKNOWN", IntegerType(), True),
    StructField("PED_CYCLIST_5_12", IntegerType(), True),
    StructField("PED_CYCLIST_13_18", IntegerType(), True),
    StructField("OLD_PED_65_AND_OVER", IntegerType(), True),
    StructField("OLD_DRIVER_75_AND_OVER", IntegerType(), True),
    StructField("YOUNG_DRIVER_18_25", IntegerType(), True),
    StructField("NO_OF_VEHICLES", IntegerType(), True),
    StructField("HEAVYVEHICLE", IntegerType(), True),
    StructField("PASSENGERVEHICLE", IntegerType(), True),
    StructField("MOTORCYCLE", IntegerType(), True),
    StructField("PT_VEHICLE", IntegerType(), True),
    StructField("DEG_URBAN_NAME", StringType(), True),
    StructField("SRNS", StringType(), True),
    StructField("RMA", StringType(), True),
    StructField("DIVIDED", StringType(), True),
    StructField("STAT_DIV_NAME", StringType(), True)
])

In [3]:
#load in csv data
file_path = "vic_road_crash_data.csv"

crashes = spark.read \
    .option("header", True) \
    .schema(schema) \
    .csv(file_path)

In [4]:
#select relevant columns for business query
crashes_selected = crashes.select(
    "ACCIDENT_NO",
    "ACCIDENT_TIME",
    "DTP_REGION",
    "ROAD_TYPE",
    "SERIOUSINJURY",
    "FATALITY"
)

crashes_selected.show(5)

+------------+-------------+-------------+---------+-------------+--------+
| ACCIDENT_NO|ACCIDENT_TIME|   DTP_REGION|ROAD_TYPE|SERIOUSINJURY|FATALITY|
+------------+-------------+-------------+---------+-------------+--------+
|T20140024624|     18:35:00|    GIPPSLAND|   STREET|            0|       0|
|T20190026336|     15:45:00|  INNER METRO|     ROAD|            0|       0|
|T20190019196|     12:07:00|  INNER METRO|     ROAD|            0|       0|
|T20250029202|     13:10:00|GREATER METRO|  HIGHWAY|            1|       0|
|T20210005363|     06:30:00|GREATER METRO|     ROAD|            1|       0|
+------------+-------------+-------------+---------+-------------+--------+
only showing top 5 rows


### Create necessary variables to conduct analysis to answer business query

In [5]:
#convert accident_time to a single hour number
crashes_time = crashes_selected.withColumn(
    "HOUR",
    substring("ACCIDENT_TIME", 1, 2).cast("int")
)

#categorise the different values of HOUR into different times of day
crashes_time = crashes_time.withColumn(
    "TIME_PERIOD",
    when(col("HOUR") < 6, "Overnight")
    .when(col("HOUR") < 12, "Morning")
    .when(col("HOUR") < 18, "Afternoon")
    .otherwise("Evening")
)

#display top rows to confirm accurate implementation
crashes_time.select(
    "ACCIDENT_TIME",
    "HOUR",
    "TIME_PERIOD"
).show(10)

+-------------+----+-----------+
|ACCIDENT_TIME|HOUR|TIME_PERIOD|
+-------------+----+-----------+
|     18:35:00|  18|    Evening|
|     15:45:00|  15|  Afternoon|
|     12:07:00|  12|  Afternoon|
|     13:10:00|  13|  Afternoon|
|     06:30:00|   6|    Morning|
|     16:55:00|  16|  Afternoon|
|     13:28:00|  13|  Afternoon|
|     13:30:00|  13|  Afternoon|
|     10:20:00|  10|    Morning|
|     15:00:00|  15|  Afternoon|
+-------------+----+-----------+
only showing top 10 rows


In [6]:
#create indicator of whether the accident resulted in 1 or more serious injuries or fatality
crashes_prepared = crashes_time.withColumn(
    "SERIOUS_FATAL_CRASH",
    when(
        (col("SERIOUSINJURY") > 0) | (col("FATALITY") > 0),
        1
    ).otherwise(0)
)

#display
crashes_prepared.select(
    "ACCIDENT_NO",
    "SERIOUSINJURY",
    "FATALITY",
    "SERIOUS_FATAL_CRASH"
).show(10)

+------------+-------------+--------+-------------------+
| ACCIDENT_NO|SERIOUSINJURY|FATALITY|SERIOUS_FATAL_CRASH|
+------------+-------------+--------+-------------------+
|T20140024624|            0|       0|                  0|
|T20190026336|            0|       0|                  0|
|T20190019196|            0|       0|                  0|
|T20250029202|            1|       0|                  1|
|T20210005363|            1|       0|                  1|
|T20210014188|            0|       0|                  0|
|T20250004689|            0|       0|                  0|
|T20200020837|            1|       0|                  1|
|T20220026535|            0|       0|                  0|
|T20240003419|            0|       0|                  0|
+------------+-------------+--------+-------------------+
only showing top 10 rows


### Cache data at this point

The dataframe crashes_prepared should be cached at this point because it is the dataset that will be used for aggregation and analysis. Caching will avoid needing to recalculate the generated fields such as time_period and serious_fatal_crash if the dataframe is reused. Aggregate results that are generated subsequently do not need to be cached as their computation is a lot smaller. For example the count statement that follows calls upon the cache, as is able to be processed quickly.

In [7]:
crashes_prepared.cache()

# Trigger the cache
crashes_prepared.count()

200352

### Aggregation

In [8]:
#record crashes by DTP region, time period, and road type
aggregated = crashes_prepared.groupBy(
    "DTP_REGION",
    "TIME_PERIOD",
    "ROAD_TYPE"
).agg(
    countDistinct("ACCIDENT_NO").alias("TOTAL_CRASHES"),
    sum("SERIOUS_FATAL_CRASH").alias("SERIOUS_FATAL_CRASHES"),
    sum("SERIOUSINJURY").alias("SERIOUS_INJURIES"),
    sum("FATALITY").alias("FATALITIES")
)

aggregated.show(10)

+-----------------+-----------+---------+-------------+---------------------+----------------+----------+
|       DTP_REGION|TIME_PERIOD|ROAD_TYPE|TOTAL_CRASHES|SERIOUS_FATAL_CRASHES|SERIOUS_INJURIES|FATALITIES|
+-----------------+-----------+---------+-------------+---------------------+----------------+----------+
|      INNER METRO|    Morning|ESPLANADE|           55|                   24|              25|         1|
|    GREATER METRO|    Evening|  HIGHWAY|         1519|                  598|             736|        28|
|    GREATER METRO|  Overnight|     LANE|            8|                    4|               4|         0|
|BARWON SOUTH WEST|    Evening|     RAMP|           12|                    6|               8|         0|
|    LODDON MALLEE|  Overnight|    PLACE|            1|                    1|               1|         0|
|      INNER METRO|  Afternoon|     NULL|          539|                  183|             208|         3|
|             HUME|    Evening|     NULL|     

### Calculate serious/fatal crash rate

In [9]:
#create ratio using number of crashes resulting in injury or death over total number of crashes 
results = aggregated.withColumn(
    "SERIOUS_FATAL_RATE",
    round(
        col("SERIOUS_FATAL_CRASHES") /
        col("TOTAL_CRASHES") * 100,
        2
    )
)

results.show(10)

+-----------------+-----------+---------+-------------+---------------------+----------------+----------+------------------+
|       DTP_REGION|TIME_PERIOD|ROAD_TYPE|TOTAL_CRASHES|SERIOUS_FATAL_CRASHES|SERIOUS_INJURIES|FATALITIES|SERIOUS_FATAL_RATE|
+-----------------+-----------+---------+-------------+---------------------+----------------+----------+------------------+
|      INNER METRO|    Morning|ESPLANADE|           55|                   24|              25|         1|             43.64|
|    GREATER METRO|    Evening|  HIGHWAY|         1519|                  598|             736|        28|             39.37|
|    GREATER METRO|  Overnight|     LANE|            8|                    4|               4|         0|              50.0|
|BARWON SOUTH WEST|    Evening|     RAMP|           12|                    6|               8|         0|              50.0|
|    LODDON MALLEE|  Overnight|    PLACE|            1|                    1|               1|         0|             100.0|


### Remove groups that contain less than 100 crashes

In [10]:
filtered_results = results.filter(
    col("TOTAL_CRASHES") >= 100
)

filtered_results.show(10)

+-----------------+-----------+---------+-------------+---------------------+----------------+----------+------------------+
|       DTP_REGION|TIME_PERIOD|ROAD_TYPE|TOTAL_CRASHES|SERIOUS_FATAL_CRASHES|SERIOUS_INJURIES|FATALITIES|SERIOUS_FATAL_RATE|
+-----------------+-----------+---------+-------------+---------------------+----------------+----------+------------------+
|    GREATER METRO|    Evening|  HIGHWAY|         1519|                  598|             736|        28|             39.37|
|      INNER METRO|  Afternoon|     NULL|          539|                  183|             208|         3|             33.95|
|        GRAMPIANS|    Evening|     ROAD|          765|                  407|             469|        35|              53.2|
|             HUME|  Afternoon|  HIGHWAY|          699|                  356|             452|        49|             50.93|
|BARWON SOUTH WEST|  Afternoon|    DRIVE|          104|                   46|              48|         0|             44.23|


### Rank DTP + ROAD_TYPE in descending serious/fatal crash rate

In [11]:
#create window and partition data by DTP region
window_spec = Window \
    .partitionBy("DTP_REGION") \
    .orderBy(col("SERIOUS_FATAL_RATE").desc())

#create field that ranks DTP and road_type by the crash rate
ranked_results = filtered_results.withColumn(
    "REGION_RANK",
    row_number().over(window_spec)
)

ranked_results.show(10)

+-----------------+-----------+---------+-------------+---------------------+----------------+----------+------------------+-----------+
|       DTP_REGION|TIME_PERIOD|ROAD_TYPE|TOTAL_CRASHES|SERIOUS_FATAL_CRASHES|SERIOUS_INJURIES|FATALITIES|SERIOUS_FATAL_RATE|REGION_RANK|
+-----------------+-----------+---------+-------------+---------------------+----------------+----------+------------------+-----------+
|BARWON SOUTH WEST|  Overnight|  HIGHWAY|          154|                  103|             109|        11|             66.88|          1|
|BARWON SOUTH WEST|  Overnight|     ROAD|          448|                  271|             295|        31|             60.49|          2|
|BARWON SOUTH WEST|    Evening|  HIGHWAY|          389|                  218|             281|        20|             56.04|          3|
|BARWON SOUTH WEST|    Evening|     ROAD|         1274|                  704|             907|        41|             55.26|          4|
|BARWON SOUTH WEST|    Morning|     ROAD|

### Generate top 3 worst road types at a particular time period when considering the serious/fatal crash rate

In [12]:
#filter for top 3 from each region
top_results = ranked_results.filter(
    col("REGION_RANK") <= 3
)

#generate summary output
final_results = top_results.select(
    "DTP_REGION",
    "TIME_PERIOD",
    "ROAD_TYPE",
    "TOTAL_CRASHES",
    "SERIOUS_FATAL_CRASHES",
    "SERIOUS_INJURIES",
    "FATALITIES",
    "SERIOUS_FATAL_RATE",
    "REGION_RANK"
).orderBy(
    "DTP_REGION",
    "REGION_RANK"
)

#show output
final_results.show(25)

+-----------------+-----------+---------+-------------+---------------------+----------------+----------+------------------+-----------+
|       DTP_REGION|TIME_PERIOD|ROAD_TYPE|TOTAL_CRASHES|SERIOUS_FATAL_CRASHES|SERIOUS_INJURIES|FATALITIES|SERIOUS_FATAL_RATE|REGION_RANK|
+-----------------+-----------+---------+-------------+---------------------+----------------+----------+------------------+-----------+
|BARWON SOUTH WEST|  Overnight|  HIGHWAY|          154|                  103|             109|        11|             66.88|          1|
|BARWON SOUTH WEST|  Overnight|     ROAD|          448|                  271|             295|        31|             60.49|          2|
|BARWON SOUTH WEST|    Evening|  HIGHWAY|          389|                  218|             281|        20|             56.04|          3|
|        GIPPSLAND|  Overnight|     ROAD|          370|                  190|             196|        28|             51.35|          1|
|        GIPPSLAND|  Overnight|  HIGHWAY|

### View execution plan

In [13]:
final_results.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (23)
+- Sort (22)
   +- Exchange (21)
      +- Filter (20)
         +- Window (19)
            +- WindowGroupLimit (18)
               +- Sort (17)
                  +- Exchange (16)
                     +- WindowGroupLimit (15)
                        +- Sort (14)
                           +- Project (13)
                              +- Filter (12)
                                 +- HashAggregate (11)
                                    +- Exchange (10)
                                       +- HashAggregate (9)
                                          +- HashAggregate (8)
                                             +- Exchange (7)
                                                +- HashAggregate (6)
                                                   +- InMemoryTableScan (1)
                                                         +- InMemoryRelation (2)
                                                               +- * Project (5)
          

### Optimisation discussion

It was important to use select() as a way to project in this instance because there are a lot of variables in the dataset that are not necessary to the analysis required to answer the business query. This demonstates Catalyst optimisation as the select avoids carrying unnecessary variables from the CSV, only selecting the ones that are required for aggregation. 

The variables 1. serious/fatal crash indicator and 2. the time period transformation are done prior to aggregation, whereas filtering for a small volume of crashes is done after aggregation from the smaller aggregated dataset. This means that predicate pushdown is limited in this query as the filtering was done post aggregation. Caching is done on the dataset that contains all data required for the business query anaylsis. No joining is necessary in this instance as all necessary variables are present in the Victorian Crash Data table. This is good as joins would introduce potentially unnecessary data movement. Spark determines an efficient way to execute all these tasks so that there is very little unnecessary data movement and aggregation is done in an optimised way.

## 3. Spark SQL Implementation

### Create temp view of data

In [13]:
#create temp view
crashes_selected.createOrReplaceTempView("crashes")

#verify view
spark.sql("SELECT * FROM crashes LIMIT 5").show()

+------------+-------------+-------------+---------+-------------+--------+
| ACCIDENT_NO|ACCIDENT_TIME|   DTP_REGION|ROAD_TYPE|SERIOUSINJURY|FATALITY|
+------------+-------------+-------------+---------+-------------+--------+
|T20140024624|     18:35:00|    GIPPSLAND|   STREET|            0|       0|
|T20190026336|     15:45:00|  INNER METRO|     ROAD|            0|       0|
|T20190019196|     12:07:00|  INNER METRO|     ROAD|            0|       0|
|T20250029202|     13:10:00|GREATER METRO|  HIGHWAY|            1|       0|
|T20210005363|     06:30:00|GREATER METRO|     ROAD|            1|       0|
+------------+-------------+-------------+---------+-------------+--------+



26/09/14 20:06:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


### SQL query to answer business query

In [14]:
#time_data CTE creates HOUR to eventually create TIME_PERIOD
#prepared_data creates TIME_PERIOD and SERIOUS_FATAL_CRASH flags 
#aggregated_data creates summary data including SERIOUS_FATAL_RATE, and filters for COUNT(ACCIDENT_NO) >= 100
#ranked_data orders the DTP_REGION by SERIOUS_FATAL_RATE and creates rank
#final select presents only top 3 TIME_PERIOD, ROAD_TYPE for each DTP_REGION in descending order 

sql_query = """
WITH time_data AS (
    SELECT
        ACCIDENT_NO,
        DTP_REGION,
        ROAD_TYPE,
        SERIOUSINJURY,
        FATALITY,
        CAST(SUBSTRING(ACCIDENT_TIME, 1, 2) AS INT) AS HOUR
    FROM crashes
),

prepared_data AS (
    SELECT
        ACCIDENT_NO,
        DTP_REGION,
        ROAD_TYPE,
        SERIOUSINJURY,
        FATALITY,
        CASE
            WHEN HOUR < 6 THEN 'Overnight'
            WHEN HOUR < 12 THEN 'Morning'
            WHEN HOUR < 18 THEN 'Afternoon'
            ELSE 'Evening'
        END AS TIME_PERIOD,
        CASE
            WHEN SERIOUSINJURY > 0 OR FATALITY > 0 THEN 1
            ELSE 0
        END AS SERIOUS_FATAL_CRASH
    FROM time_data
),

aggregated_data AS (
    SELECT
        DTP_REGION,
        TIME_PERIOD,
        ROAD_TYPE,
        COUNT(ACCIDENT_NO) AS TOTAL_CRASHES,
        SUM(SERIOUS_FATAL_CRASH) AS SERIOUS_FATAL_CRASHES,
        SUM(SERIOUSINJURY) AS SERIOUS_INJURIES,
        SUM(FATALITY) AS FATALITIES,
        ROUND(
            SUM(SERIOUS_FATAL_CRASH) / COUNT(ACCIDENT_NO) * 100,
            2
        ) AS SERIOUS_FATAL_RATE
    FROM prepared_data
    GROUP BY
        DTP_REGION,
        TIME_PERIOD,
        ROAD_TYPE
    HAVING COUNT(ACCIDENT_NO) >= 100
),

ranked_data AS (
    SELECT
        DTP_REGION,
        TIME_PERIOD,
        ROAD_TYPE,
        TOTAL_CRASHES,
        SERIOUS_FATAL_CRASHES,
        SERIOUS_INJURIES,
        FATALITIES,
        SERIOUS_FATAL_RATE,
        ROW_NUMBER() OVER (
            PARTITION BY DTP_REGION
            ORDER BY SERIOUS_FATAL_RATE DESC
        ) AS REGION_RANK
    FROM aggregated_data
)

SELECT *
FROM ranked_data
WHERE REGION_RANK <= 3
ORDER BY
    DTP_REGION,
    REGION_RANK
"""

In [15]:
sql_results = spark.sql(sql_query)

sql_results.show(25)

+-----------------+-----------+---------+-------------+---------------------+----------------+----------+------------------+-----------+
|       DTP_REGION|TIME_PERIOD|ROAD_TYPE|TOTAL_CRASHES|SERIOUS_FATAL_CRASHES|SERIOUS_INJURIES|FATALITIES|SERIOUS_FATAL_RATE|REGION_RANK|
+-----------------+-----------+---------+-------------+---------------------+----------------+----------+------------------+-----------+
|BARWON SOUTH WEST|  Overnight|  HIGHWAY|          154|                  103|             109|        11|             66.88|          1|
|BARWON SOUTH WEST|  Overnight|     ROAD|          448|                  271|             295|        31|             60.49|          2|
|BARWON SOUTH WEST|    Evening|  HIGHWAY|          389|                  218|             281|        20|             56.04|          3|
|        GIPPSLAND|  Overnight|     ROAD|          370|                  190|             196|        28|             51.35|          1|
|        GIPPSLAND|  Overnight|  HIGHWAY|

### DataFrame and Spark SQL  differences discussion (readability, maintainability, query structure)

The spark dataframe and spark SQL codes produce the same results as they perform the same operations. The dataframe implementation might be considered easier to follow as the commands are more closely linked to the data, such as groupBy() and agg(). The spark SQL implementation is more stuctured with the use of CTEs to generate and complete different parts of the business query analysis (one CTE to create the required feature, another to aggregate the data, another to rank, etc.). Both queries are maintainable, the dataframe one if the analyst is more familiar with python, and the SQL one is better if the operations required are more relational (select, join, etc.).

The SQL implementation structures the operations through the CTES time_data, prepared_data, aggregated_data, and ranked_data, so it makes the process from raw data to final results clear and aligned to the business query. The dataframe does the same using chained python transformations, which could provide more flexibility if the transformations need to be constructed dynamically.

For debuggability, you could argue the SQL implementation is better since the CTEs allow for easier identification of whether the hour extraction is appropriate by inspecting prepared_data.

Overall, the SQL approach is good since the analysis calls for mainly relational operations, while the dataframe integrates more reusable functions for better python-based processing that might be required.

## 4. Result Validation

### Compare number of results

In [17]:
print("DataFrame API rows:", final_results.count())
print("Spark SQL rows:", sql_results.count())

DataFrame API rows: 21
Spark SQL rows: 21


### Compare results

In [16]:
df_api = final_results.orderBy(
    "DTP_REGION",
    "REGION_RANK"
)

df_sql = sql_results.orderBy(
    "DTP_REGION",
    "REGION_RANK"
)

df_api.show(21)
df_sql.show(21)

+-----------------+-----------+---------+-------------+---------------------+----------------+----------+------------------+-----------+
|       DTP_REGION|TIME_PERIOD|ROAD_TYPE|TOTAL_CRASHES|SERIOUS_FATAL_CRASHES|SERIOUS_INJURIES|FATALITIES|SERIOUS_FATAL_RATE|REGION_RANK|
+-----------------+-----------+---------+-------------+---------------------+----------------+----------+------------------+-----------+
|BARWON SOUTH WEST|  Overnight|  HIGHWAY|          154|                  103|             109|        11|             66.88|          1|
|BARWON SOUTH WEST|  Overnight|     ROAD|          448|                  271|             295|        31|             60.49|          2|
|BARWON SOUTH WEST|    Evening|  HIGHWAY|          389|                  218|             281|        20|             56.04|          3|
|        GIPPSLAND|  Overnight|     ROAD|          370|                  190|             196|        28|             51.35|          1|
|        GIPPSLAND|  Overnight|  HIGHWAY|

### Discuss differences in results

There are no differences in results between the dataframes and SQL implementations.

Both implementations perform the same sequence of operations with extraction of the accident hour, classification into time periods, creation of the serious/fatal crash indicator, aggregation by DTP_REGION, TIME_PERIOD and ROAD_TYPE, post-aggregation filtering, calculation of the serious/fatal crash rate, and ranking within each region. The equivalent input data and transformation logic should produce equivalent results.

It is important to make sure the ORDER BY is applied at an equivalent point in the query as this could produce different results, but that was already done in the above queries. Same for numerical formatting; this could produce different results based on how the operations are applied as there could be differences in how division and decimal point rounding is applied by each of the implementations. Both of these instances are relatively minor differences if they were present, it is more about how the data output is presented.

# Part B: System perspective and performance analysis

## 1. Partition Strategy

In [17]:
#start with partition of 10
n = 10

### 1.1 Hash Partition

In [18]:
#hash partition function
hash_df = crashes.repartition(n, "ACCIDENT_NO")

#create summary of count in each partition
hash_partition_counts = hash_df \
    .withColumn("PARTITION_ID", spark_partition_id()) \
    .groupBy("PARTITION_ID") \
    .agg(count("*").alias("RECORD_COUNT")) \
    .orderBy("PARTITION_ID")

#display partition and number of records in each
hash_partition_counts.show()

+------------+------------+
|PARTITION_ID|RECORD_COUNT|
+------------+------------+
|           0|       19910|
|           1|       20005|
|           2|       19876|
|           3|       19903|
|           4|       19990|
|           5|       20312|
|           6|       20065|
|           7|       20206|
|           8|       20150|
|           9|       19935|
+------------+------------+



### 1.2 Range Partition

In [19]:
#partition by range method
range_df = crashes.repartitionByRange(n, "ACCIDENT_NO")

#create summary of count in each partition
range_partition_counts = range_df \
    .withColumn("PARTITION_ID", spark_partition_id()) \
    .groupBy("PARTITION_ID") \
    .agg(count("*").alias("RECORD_COUNT")) \
    .orderBy("PARTITION_ID")

#display partition and number of records in each
range_partition_counts.show()

+------------+------------+
|PARTITION_ID|RECORD_COUNT|
+------------+------------+
|           0|       19887|
|           1|       19927|
|           2|       19206|
|           3|       20467|
|           4|       19517|
|           5|       18158|
|           6|       21410|
|           7|       20064|
|           8|       20050|
|           9|       21666|
+------------+------------+



### 1.3 Comparative Analysis

In [20]:
#rename hash partition count table
hash_counts = hash_partition_counts \
    .withColumnRenamed("RECORD_COUNT", "HASH_RECORD_COUNT")

#rename range partition count table
range_counts = range_partition_counts \
    .withColumnRenamed("RECORD_COUNT", "RANGE_RECORD_COUNT")

#join volumes in each partition between partition method
partition_comparison = hash_counts.join(
    range_counts,
    on="PARTITION_ID",
    how="outer"
).orderBy("PARTITION_ID")

#show comparison table
partition_comparison.show()

#compute general stats for hash partition
hash_stats = hash_partition_counts.agg(
    min("RECORD_COUNT").alias("MIN_RECORDS"),
    max("RECORD_COUNT").alias("MAX_RECORDS"),
    avg("RECORD_COUNT").alias("AVG_RECORDS")
)

#compute general stats for range partition
range_stats = range_partition_counts.agg(
    min("RECORD_COUNT").alias("MIN_RECORDS"),
    max("RECORD_COUNT").alias("MAX_RECORDS"),
    avg("RECORD_COUNT").alias("AVG_RECORDS")
)

print("Hash partition statistics:")
hash_stats.show()

print("Range partition statistics:")
range_stats.show()

+------------+-----------------+------------------+
|PARTITION_ID|HASH_RECORD_COUNT|RANGE_RECORD_COUNT|
+------------+-----------------+------------------+
|           0|            19910|             18334|
|           1|            20005|             19717|
|           2|            19876|             19444|
|           3|            19903|             21960|
|           4|            19990|             19847|
|           5|            20312|             18855|
|           6|            20065|             21847|
|           7|            20206|             20558|
|           8|            20150|             19820|
|           9|            19935|             19970|
+------------+-----------------+------------------+

Hash partition statistics:
+-----------+-----------+-----------+
|MIN_RECORDS|MAX_RECORDS|AVG_RECORDS|
+-----------+-----------+-----------+
|      19876|      20312|    20035.2|
+-----------+-----------+-----------+

Range partition statistics:
+-----------+-----------+

Hash and range partitioning were completed using the high cardinality variable ACCIDENT_NO, and 10 partitions were created. This means that each partition should have around 20,000 observations present, which is a good balance to observe the distribution of data in each one and is not intense when running using spark.

From the summary statistics, it appears that hash partitioning is more even, with the difference between the min_records and max_records being more narrow that the range partition statistics. Both methods produce the same average of 20035 records though, and this is because the number of total rows in the data and the number of partitions were consistent between the methods. 

The hash partitioning shows very little evidence of data skew since the parition sizes are very similar to one another, while the range partition gives more observations to some partitions compared to others. The range is not extremely different so this probably isn't evident of severe skew. This is because ACCIDENT_NO values might not be evenly distributed across the ranges selected by the range partitioning. If there was skew present through another partitioning strategy, hash partitioning would be a suitable mitigation method in this instance.

In this instance, the hash partitioning is more suitable as the result is more even and gives opportunity to more even workload across the partitions. This will likely lead to better query performance than compared to range partitioning. On the other hand, if there is a need for ordered operations, that is when range partitioning might be more useful, but it is still a little more uneven.

## 2. Execution time benchmarking

### Dataframe benchmarking

In [26]:
%%time

benchmark_df = crashes_selected \
    .withColumn(
        "HOUR",
        substring("ACCIDENT_TIME", 1, 2).cast("int")
    ) \
    .withColumn(
        "TIME_PERIOD",
        when(col("HOUR") < 6, "Overnight")
        .when(col("HOUR") < 12, "Morning")
        .when(col("HOUR") < 18, "Afternoon")
        .otherwise("Evening")
    ) \
    .withColumn(
        "SERIOUS_FATAL_CRASH",
        when(
            (col("SERIOUSINJURY") > 0) | (col("FATALITY") > 0),
            1
        ).otherwise(0)
    ) \
    .groupBy(
        "DTP_REGION",
        "TIME_PERIOD",
        "ROAD_TYPE"
    ) \
    .agg(
        count("ACCIDENT_NO").alias("TOTAL_CRASHES"),
        sum("SERIOUS_FATAL_CRASH").alias("SERIOUS_FATAL_CRASHES"),
        sum("SERIOUSINJURY").alias("SERIOUS_INJURIES"),
        sum("FATALITY").alias("FATALITIES")
    ) \
    .withColumn(
        "SERIOUS_FATAL_RATE",
        round(
            col("SERIOUS_FATAL_CRASHES") /
            col("TOTAL_CRASHES") * 100,
            2
        )
    ) \
    .filter(col("TOTAL_CRASHES") >= 100)

window_spec = Window \
    .partitionBy("DTP_REGION") \
    .orderBy(col("SERIOUS_FATAL_RATE").desc())

benchmark_df = benchmark_df.withColumn(
    "REGION_RANK",
    row_number().over(window_spec)
)

benchmark_df = benchmark_df \
    .filter(col("REGION_RANK") <= 3) \
    .orderBy("DTP_REGION", "REGION_RANK")

benchmark_df.collect()

CPU times: user 11.2 ms, sys: 5.86 ms, total: 17.1 ms
Wall time: 324 ms


[Row(DTP_REGION='BARWON SOUTH WEST', TIME_PERIOD='Overnight', ROAD_TYPE='HIGHWAY', TOTAL_CRASHES=154, SERIOUS_FATAL_CRASHES=103, SERIOUS_INJURIES=109, FATALITIES=11, SERIOUS_FATAL_RATE=66.88, REGION_RANK=1),
 Row(DTP_REGION='BARWON SOUTH WEST', TIME_PERIOD='Overnight', ROAD_TYPE='ROAD', TOTAL_CRASHES=448, SERIOUS_FATAL_CRASHES=271, SERIOUS_INJURIES=295, FATALITIES=31, SERIOUS_FATAL_RATE=60.49, REGION_RANK=2),
 Row(DTP_REGION='BARWON SOUTH WEST', TIME_PERIOD='Evening', ROAD_TYPE='HIGHWAY', TOTAL_CRASHES=389, SERIOUS_FATAL_CRASHES=218, SERIOUS_INJURIES=281, FATALITIES=20, SERIOUS_FATAL_RATE=56.04, REGION_RANK=3),
 Row(DTP_REGION='GIPPSLAND', TIME_PERIOD='Overnight', ROAD_TYPE='ROAD', TOTAL_CRASHES=370, SERIOUS_FATAL_CRASHES=190, SERIOUS_INJURIES=196, FATALITIES=28, SERIOUS_FATAL_RATE=51.35, REGION_RANK=1),
 Row(DTP_REGION='GIPPSLAND', TIME_PERIOD='Overnight', ROAD_TYPE='HIGHWAY', TOTAL_CRASHES=113, SERIOUS_FATAL_CRASHES=53, SERIOUS_INJURIES=51, FATALITIES=11, SERIOUS_FATAL_RATE=46.9, REG

In [24]:
%%time

benchmark_df = crashes_selected \
    .withColumn(
        "HOUR",
        substring("ACCIDENT_TIME", 1, 2).cast("int")
    ) \
    .withColumn(
        "TIME_PERIOD",
        when(col("HOUR") < 6, "Overnight")
        .when(col("HOUR") < 12, "Morning")
        .when(col("HOUR") < 18, "Afternoon")
        .otherwise("Evening")
    ) \
    .withColumn(
        "SERIOUS_FATAL_CRASH",
        when(
            (col("SERIOUSINJURY") > 0) | (col("FATALITY") > 0),
            1
        ).otherwise(0)
    ) \
    .groupBy(
        "DTP_REGION",
        "TIME_PERIOD",
        "ROAD_TYPE"
    ) \
    .agg(
        count("ACCIDENT_NO").alias("TOTAL_CRASHES"),
        sum("SERIOUS_FATAL_CRASH").alias("SERIOUS_FATAL_CRASHES"),
        sum("SERIOUSINJURY").alias("SERIOUS_INJURIES"),
        sum("FATALITY").alias("FATALITIES")
    ) \
    .withColumn(
        "SERIOUS_FATAL_RATE",
        round(
            col("SERIOUS_FATAL_CRASHES") /
            col("TOTAL_CRASHES") * 100,
            2
        )
    ) \
    .filter(col("TOTAL_CRASHES") >= 100)

window_spec = Window \
    .partitionBy("DTP_REGION") \
    .orderBy(col("SERIOUS_FATAL_RATE").desc())

benchmark_df = benchmark_df.withColumn(
    "REGION_RANK",
    row_number().over(window_spec)
)

benchmark_df = benchmark_df \
    .filter(col("REGION_RANK") <= 3) \
    .orderBy("DTP_REGION", "REGION_RANK")

benchmark_df.collect()

CPU times: user 10.6 ms, sys: 6.43 ms, total: 17 ms
Wall time: 375 ms


[Row(DTP_REGION='BARWON SOUTH WEST', TIME_PERIOD='Overnight', ROAD_TYPE='HIGHWAY', TOTAL_CRASHES=154, SERIOUS_FATAL_CRASHES=103, SERIOUS_INJURIES=109, FATALITIES=11, SERIOUS_FATAL_RATE=66.88, REGION_RANK=1),
 Row(DTP_REGION='BARWON SOUTH WEST', TIME_PERIOD='Overnight', ROAD_TYPE='ROAD', TOTAL_CRASHES=448, SERIOUS_FATAL_CRASHES=271, SERIOUS_INJURIES=295, FATALITIES=31, SERIOUS_FATAL_RATE=60.49, REGION_RANK=2),
 Row(DTP_REGION='BARWON SOUTH WEST', TIME_PERIOD='Evening', ROAD_TYPE='HIGHWAY', TOTAL_CRASHES=389, SERIOUS_FATAL_CRASHES=218, SERIOUS_INJURIES=281, FATALITIES=20, SERIOUS_FATAL_RATE=56.04, REGION_RANK=3),
 Row(DTP_REGION='GIPPSLAND', TIME_PERIOD='Overnight', ROAD_TYPE='ROAD', TOTAL_CRASHES=370, SERIOUS_FATAL_CRASHES=190, SERIOUS_INJURIES=196, FATALITIES=28, SERIOUS_FATAL_RATE=51.35, REGION_RANK=1),
 Row(DTP_REGION='GIPPSLAND', TIME_PERIOD='Overnight', ROAD_TYPE='HIGHWAY', TOTAL_CRASHES=113, SERIOUS_FATAL_CRASHES=53, SERIOUS_INJURIES=51, FATALITIES=11, SERIOUS_FATAL_RATE=46.9, REG

In [25]:
%%time

benchmark_df = crashes_selected \
    .withColumn(
        "HOUR",
        substring("ACCIDENT_TIME", 1, 2).cast("int")
    ) \
    .withColumn(
        "TIME_PERIOD",
        when(col("HOUR") < 6, "Overnight")
        .when(col("HOUR") < 12, "Morning")
        .when(col("HOUR") < 18, "Afternoon")
        .otherwise("Evening")
    ) \
    .withColumn(
        "SERIOUS_FATAL_CRASH",
        when(
            (col("SERIOUSINJURY") > 0) | (col("FATALITY") > 0),
            1
        ).otherwise(0)
    ) \
    .groupBy(
        "DTP_REGION",
        "TIME_PERIOD",
        "ROAD_TYPE"
    ) \
    .agg(
        count("ACCIDENT_NO").alias("TOTAL_CRASHES"),
        sum("SERIOUS_FATAL_CRASH").alias("SERIOUS_FATAL_CRASHES"),
        sum("SERIOUSINJURY").alias("SERIOUS_INJURIES"),
        sum("FATALITY").alias("FATALITIES")
    ) \
    .withColumn(
        "SERIOUS_FATAL_RATE",
        round(
            col("SERIOUS_FATAL_CRASHES") /
            col("TOTAL_CRASHES") * 100,
            2
        )
    ) \
    .filter(col("TOTAL_CRASHES") >= 100)

window_spec = Window \
    .partitionBy("DTP_REGION") \
    .orderBy(col("SERIOUS_FATAL_RATE").desc())

benchmark_df = benchmark_df.withColumn(
    "REGION_RANK",
    row_number().over(window_spec)
)

benchmark_df = benchmark_df \
    .filter(col("REGION_RANK") <= 3) \
    .orderBy("DTP_REGION", "REGION_RANK")

benchmark_df.collect()

CPU times: user 9.64 ms, sys: 5.76 ms, total: 15.4 ms
Wall time: 330 ms


[Row(DTP_REGION='BARWON SOUTH WEST', TIME_PERIOD='Overnight', ROAD_TYPE='HIGHWAY', TOTAL_CRASHES=154, SERIOUS_FATAL_CRASHES=103, SERIOUS_INJURIES=109, FATALITIES=11, SERIOUS_FATAL_RATE=66.88, REGION_RANK=1),
 Row(DTP_REGION='BARWON SOUTH WEST', TIME_PERIOD='Overnight', ROAD_TYPE='ROAD', TOTAL_CRASHES=448, SERIOUS_FATAL_CRASHES=271, SERIOUS_INJURIES=295, FATALITIES=31, SERIOUS_FATAL_RATE=60.49, REGION_RANK=2),
 Row(DTP_REGION='BARWON SOUTH WEST', TIME_PERIOD='Evening', ROAD_TYPE='HIGHWAY', TOTAL_CRASHES=389, SERIOUS_FATAL_CRASHES=218, SERIOUS_INJURIES=281, FATALITIES=20, SERIOUS_FATAL_RATE=56.04, REGION_RANK=3),
 Row(DTP_REGION='GIPPSLAND', TIME_PERIOD='Overnight', ROAD_TYPE='ROAD', TOTAL_CRASHES=370, SERIOUS_FATAL_CRASHES=190, SERIOUS_INJURIES=196, FATALITIES=28, SERIOUS_FATAL_RATE=51.35, REGION_RANK=1),
 Row(DTP_REGION='GIPPSLAND', TIME_PERIOD='Overnight', ROAD_TYPE='HIGHWAY', TOTAL_CRASHES=113, SERIOUS_FATAL_CRASHES=53, SERIOUS_INJURIES=51, FATALITIES=11, SERIOUS_FATAL_RATE=46.9, REG

### SQL Benchmarking

In [30]:
%%time

sql_benchmark = spark.sql(sql_query)

sql_benchmark.collect()

CPU times: user 1.51 ms, sys: 1.19 ms, total: 2.7 ms
Wall time: 445 ms


[Row(DTP_REGION='BARWON SOUTH WEST', TIME_PERIOD='Overnight', ROAD_TYPE='HIGHWAY', TOTAL_CRASHES=154, SERIOUS_FATAL_CRASHES=103, SERIOUS_INJURIES=109, FATALITIES=11, SERIOUS_FATAL_RATE=66.88, REGION_RANK=1),
 Row(DTP_REGION='BARWON SOUTH WEST', TIME_PERIOD='Overnight', ROAD_TYPE='ROAD', TOTAL_CRASHES=448, SERIOUS_FATAL_CRASHES=271, SERIOUS_INJURIES=295, FATALITIES=31, SERIOUS_FATAL_RATE=60.49, REGION_RANK=2),
 Row(DTP_REGION='BARWON SOUTH WEST', TIME_PERIOD='Evening', ROAD_TYPE='HIGHWAY', TOTAL_CRASHES=389, SERIOUS_FATAL_CRASHES=218, SERIOUS_INJURIES=281, FATALITIES=20, SERIOUS_FATAL_RATE=56.04, REGION_RANK=3),
 Row(DTP_REGION='GIPPSLAND', TIME_PERIOD='Overnight', ROAD_TYPE='ROAD', TOTAL_CRASHES=370, SERIOUS_FATAL_CRASHES=190, SERIOUS_INJURIES=196, FATALITIES=28, SERIOUS_FATAL_RATE=51.35, REGION_RANK=1),
 Row(DTP_REGION='GIPPSLAND', TIME_PERIOD='Overnight', ROAD_TYPE='HIGHWAY', TOTAL_CRASHES=113, SERIOUS_FATAL_CRASHES=53, SERIOUS_INJURIES=51, FATALITIES=11, SERIOUS_FATAL_RATE=46.9, REG

In [31]:
%%time

sql_benchmark = spark.sql(sql_query)

# Force complete execution
sql_benchmark.collect()

CPU times: user 1.76 ms, sys: 767 μs, total: 2.53 ms
Wall time: 342 ms


[Row(DTP_REGION='BARWON SOUTH WEST', TIME_PERIOD='Overnight', ROAD_TYPE='HIGHWAY', TOTAL_CRASHES=154, SERIOUS_FATAL_CRASHES=103, SERIOUS_INJURIES=109, FATALITIES=11, SERIOUS_FATAL_RATE=66.88, REGION_RANK=1),
 Row(DTP_REGION='BARWON SOUTH WEST', TIME_PERIOD='Overnight', ROAD_TYPE='ROAD', TOTAL_CRASHES=448, SERIOUS_FATAL_CRASHES=271, SERIOUS_INJURIES=295, FATALITIES=31, SERIOUS_FATAL_RATE=60.49, REGION_RANK=2),
 Row(DTP_REGION='BARWON SOUTH WEST', TIME_PERIOD='Evening', ROAD_TYPE='HIGHWAY', TOTAL_CRASHES=389, SERIOUS_FATAL_CRASHES=218, SERIOUS_INJURIES=281, FATALITIES=20, SERIOUS_FATAL_RATE=56.04, REGION_RANK=3),
 Row(DTP_REGION='GIPPSLAND', TIME_PERIOD='Overnight', ROAD_TYPE='ROAD', TOTAL_CRASHES=370, SERIOUS_FATAL_CRASHES=190, SERIOUS_INJURIES=196, FATALITIES=28, SERIOUS_FATAL_RATE=51.35, REGION_RANK=1),
 Row(DTP_REGION='GIPPSLAND', TIME_PERIOD='Overnight', ROAD_TYPE='HIGHWAY', TOTAL_CRASHES=113, SERIOUS_FATAL_CRASHES=53, SERIOUS_INJURIES=51, FATALITIES=11, SERIOUS_FATAL_RATE=46.9, REG

In [32]:
%%time

sql_benchmark = spark.sql(sql_query)

# Force complete execution
sql_benchmark.collect()

CPU times: user 1.58 ms, sys: 822 μs, total: 2.41 ms
Wall time: 346 ms


[Row(DTP_REGION='BARWON SOUTH WEST', TIME_PERIOD='Overnight', ROAD_TYPE='HIGHWAY', TOTAL_CRASHES=154, SERIOUS_FATAL_CRASHES=103, SERIOUS_INJURIES=109, FATALITIES=11, SERIOUS_FATAL_RATE=66.88, REGION_RANK=1),
 Row(DTP_REGION='BARWON SOUTH WEST', TIME_PERIOD='Overnight', ROAD_TYPE='ROAD', TOTAL_CRASHES=448, SERIOUS_FATAL_CRASHES=271, SERIOUS_INJURIES=295, FATALITIES=31, SERIOUS_FATAL_RATE=60.49, REGION_RANK=2),
 Row(DTP_REGION='BARWON SOUTH WEST', TIME_PERIOD='Evening', ROAD_TYPE='HIGHWAY', TOTAL_CRASHES=389, SERIOUS_FATAL_CRASHES=218, SERIOUS_INJURIES=281, FATALITIES=20, SERIOUS_FATAL_RATE=56.04, REGION_RANK=3),
 Row(DTP_REGION='GIPPSLAND', TIME_PERIOD='Overnight', ROAD_TYPE='ROAD', TOTAL_CRASHES=370, SERIOUS_FATAL_CRASHES=190, SERIOUS_INJURIES=196, FATALITIES=28, SERIOUS_FATAL_RATE=51.35, REGION_RANK=1),
 Row(DTP_REGION='GIPPSLAND', TIME_PERIOD='Overnight', ROAD_TYPE='HIGHWAY', TOTAL_CRASHES=113, SERIOUS_FATAL_CRASHES=53, SERIOUS_INJURIES=51, FATALITIES=11, SERIOUS_FATAL_RATE=46.9, REG

### Summary Table

In [33]:
import statistics

df_times = [324, 375, 330]
sql_times = [445, 342, 346]

print("DataFrame median (ms):", statistics.median(df_times))
print("Spark SQL median (ms):", statistics.median(sql_times))

DataFrame median: 330
Spark SQL median: 346


### Discussion 

Benchmarking was done using Apache Spark 4.2.0 locally on a single machine, so was done using local CPU resources. Spark default memory configuration was used provide by the environment. Since the experiments were run locally, the results may be different compared to if a distributed cluster was used to execute the query.

The execution times show that the Dataframe API was slightly faster than the SQL implementation in the local environment. The median Dataframe runtime was 330ms while the SQL median is 346ms, which is a difference of 16ms, or the Dataframe implementation is 4.6% faster than that of the SQL implementation. The first SQL runtime was substantially slower at 445ms which could be due to JVM warm-up, so it was important to execute the implementation more than once and take the median.

Both the SQL and Dataframe implementations are processed through the Spark Catalyst optimiser, but there can be differences in performance that are due to query construction, optimisation, execution planning and runtime overhead. Both implementations were designed to perform similar transformations, including aggregation, filtering, window operation, and sorting, so they should generate comparable workloads. The difference between the median execution time shows that neither implementation has a big performance advantage over the other. Also, since the execution was done in a local environment, it cannot be representative of a large scale performance. Shuffle volume, partitioning, serialisation and resources have a bigger impact when executed on a cluster.

## 3. Execution plan interpretation

In [21]:
aggregated.explain(extended=True)

== Parsed Logical Plan ==
'Aggregate ['DTP_REGION, 'TIME_PERIOD, 'ROAD_TYPE], ['DTP_REGION, 'TIME_PERIOD, 'ROAD_TYPE, 'count(distinct 'ACCIDENT_NO) AS TOTAL_CRASHES#397, 'sum('SERIOUS_FATAL_CRASH) AS SERIOUS_FATAL_CRASHES#398, 'sum('SERIOUSINJURY) AS SERIOUS_INJURIES#399, 'sum('FATALITY) AS FATALITIES#400]
+- Project [ACCIDENT_NO#0, ACCIDENT_TIME#2, DTP_REGION#17, ROAD_TYPE#14, SERIOUSINJURY#25, FATALITY#24, HOUR#78, TIME_PERIOD#79, CASE WHEN ((SERIOUSINJURY#25 > 0) OR (FATALITY#24 > 0)) THEN 1 ELSE 0 END AS SERIOUS_FATAL_CRASH#91]
   +- Project [ACCIDENT_NO#0, ACCIDENT_TIME#2, DTP_REGION#17, ROAD_TYPE#14, SERIOUSINJURY#25, FATALITY#24, HOUR#78, CASE WHEN (HOUR#78 < 6) THEN Overnight WHEN (HOUR#78 < 12) THEN Morning WHEN (HOUR#78 < 18) THEN Afternoon ELSE Evening END AS TIME_PERIOD#79]
      +- Project [ACCIDENT_NO#0, ACCIDENT_TIME#2, DTP_REGION#17, ROAD_TYPE#14, SERIOUSINJURY#25, FATALITY#24, cast(substring(ACCIDENT_TIME#2, 1, 2) as int) AS HOUR#78]
         +- Project [ACCIDENT_NO#0,

### Physical Plan Annotations

The physical plan contains two exchange hashpartitioning operations and they both represent shuffle stages.

Shuffle 1 happens with the line: Exchange hashpartitioning(DTP_REGION#17, TIME_PERIOD#79, ROAD_TYPE#14, ACCIDENT_NO#0, 200). Since there is a HashAggregate right before the shuffle, this shuffle redistributes records according to the grouping columns and ACCIDENT_NO, which is required to combine records that belong to the same distinct accident and grouping combination. 

Shuffle 2 happens with the line: Exchange hashpartitioning(DTP_REGION#17, TIME_PERIOD#79, ROAD_TYPE#14, 200). The preceding operation is again the HashAggregate, and this shuffle with redistribute the aggregated records so that all that belong to the same DTP_REGION, TIME_PERIOD, ROAD_TYPE group are processed together by the final aggregation.

### Analysis

The execution plan shows the aggregation requires shuffling operation to correctly combine records across the partitions. The first Exchange hashpartitioning operator occurs after a HashAggregate where the data is partitioned using DTP_REGION, TIME_PERIOD, ROAD_TYPE, and ACCOUNT_NO and it is associated with the count(distinct ACCIDENT_NO) calculation. The second shuffle partitions the partially aggregated data using DTP_REGION, TIME_PERIOD, and ROAD_TYPE so the final aggregation can combine all records belonging to each group.

The shuffling is necessary because records that are required for the aggregation could be located in various partitions. Spark will redistribute data across partitions based on the fields that the aggregation is occurring on. This involves network I/O in a distributed cluster as data will probably need to move between executors. In a local execution environment, the shuffling doesn't involve network communication between seperate machines, but in a distributed cluster shuffle data may be transferred between executors over the network. This could create task scheduling overhead as Spark must complete the shuffle stage before the aggreation can be completed. The preceeding partial HashAggregate reduces the amount of data that needs to be shuffled though.

Completely removing the shuffling might be difficult as the operation uses multiple grouping variables and a distinct count for this analysis. The current plan portrays a necessary cost of performing the aggregation.

### 4. Spark Web UI (DAG visualitation)

In [22]:
aggregated.collect()

[Row(DTP_REGION='INNER METRO', TIME_PERIOD='Morning', ROAD_TYPE='ESPLANADE', TOTAL_CRASHES=55, SERIOUS_FATAL_CRASHES=24, SERIOUS_INJURIES=25, FATALITIES=1),
 Row(DTP_REGION='GREATER METRO', TIME_PERIOD='Evening', ROAD_TYPE='HIGHWAY', TOTAL_CRASHES=1519, SERIOUS_FATAL_CRASHES=598, SERIOUS_INJURIES=736, FATALITIES=28),
 Row(DTP_REGION='GREATER METRO', TIME_PERIOD='Overnight', ROAD_TYPE='LANE', TOTAL_CRASHES=8, SERIOUS_FATAL_CRASHES=4, SERIOUS_INJURIES=4, FATALITIES=0),
 Row(DTP_REGION='BARWON SOUTH WEST', TIME_PERIOD='Evening', ROAD_TYPE='RAMP', TOTAL_CRASHES=12, SERIOUS_FATAL_CRASHES=6, SERIOUS_INJURIES=8, FATALITIES=0),
 Row(DTP_REGION='LODDON MALLEE', TIME_PERIOD='Overnight', ROAD_TYPE='PLACE', TOTAL_CRASHES=1, SERIOUS_FATAL_CRASHES=1, SERIOUS_INJURIES=1, FATALITIES=0),
 Row(DTP_REGION='INNER METRO', TIME_PERIOD='Afternoon', ROAD_TYPE=None, TOTAL_CRASHES=539, SERIOUS_FATAL_CRASHES=183, SERIOUS_INJURIES=208, FATALITIES=3),
 Row(DTP_REGION='HUME', TIME_PERIOD='Evening', ROAD_TYPE=None, 

![DAG Screenshot](./dag.png)

The DAG shows multiple execution stages, with seperation occuring around the two Exchange operators. This represents the shuffle operations previously identified in the physical execution plan. The first one occurs after a partial HashAggregate where Spark redistributed the data according to the grouping columns and ACCIDENT_NO to get a distinct crash count. The second shuffle moves the partially aggregated data using DTP_REGION, TIME_PERIOD, and ROAD_TYPE so that the final group level aggregation can be processed. 

The DAG photo also shows AQEShuffleRead and AdaptiveSparkPlan. In a distributed cluster, the shuffle could require intermediate data to be transferred between executors, which increases network I/O and scheduling overhead. Since the application is being run locally, the shuffle occurs within the local machine instead of between physical cluster nodes.

No data skew is obvious in the DAG visualiation, which is consistent with earlier hash-partitioning that showed relatively evenly distributed volumes across the partitions. 

Potential optimisation could be in the form of reducing columns prior to aggregation, filtering data as early as possible so that partial aggregation can reduce shuffle volume. Pre-partitioning on frequently used columns could reduce redistribution for repeated workloads.